In [0]:
%pip install faker

In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------

# Create widgets for configurable parameters
dbutils.widgets.text("num_patients", "327", "Number of Patients")
dbutils.widgets.text("events_per_patient_max", "8", "Max Events Per Patient")
dbutils.widgets.text("hl7_version", "2.5", "HL7 Version")
dbutils.widgets.text("sending_app", "DATABRICKS_SIM", "Sending Application")
dbutils.widgets.text("sending_facility", "DBX_FAC", "Sending Facility")
dbutils.widgets.text("receiving_app", "DOWNSTREAM_SYS", "Receiving Application")
dbutils.widgets.text("receiving_facility", "DST_FAC", "Receiving Facility")
dbutils.widgets.text("catalog_use", "main", "Catalog Name")
dbutils.widgets.text("schema_use", "healthcare", "Schema Name")
dbutils.widgets.text("volume_name", "hl7_synthetic", "Volume Name")
dbutils.widgets.text("relative_path", "adt_fake", "Relative Path")

# Get widget values
num_patients = int(dbutils.widgets.get("num_patients"))
events_per_patient_max = int(dbutils.widgets.get("events_per_patient_max"))
hl7_version = dbutils.widgets.get("hl7_version")
sending_app = dbutils.widgets.get("sending_app")
sending_facility = dbutils.widgets.get("sending_facility")
receiving_app = dbutils.widgets.get("receiving_app")
receiving_facility = dbutils.widgets.get("receiving_facility")
catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
volume_name = dbutils.widgets.get("volume_name")
relative_path = dbutils.widgets.get("relative_path")

# Volume path: /Volumes/<catalog>/<schema>/<volume>/<path_to_file>
output_path = f"/Volumes/{catalog_use}/{schema_use}/{volume_name}/{relative_path}"

In [0]:
volume_details = spark.sql(f"""
  SELECT
    v.volume_catalog,
    v.volume_schema,
    v.volume_name,
    v.storage_location,
    e.external_location_name,
    e.url AS external_location_url
  FROM system.information_schema.volumes v
  JOIN system.information_schema.external_locations e
    -- storage_location is a subpath under the external location URL
    ON lower(v.storage_location) LIKE concat(lower(e.url), '%')
  WHERE v.volume_catalog = '{catalog_use}'
    AND v.volume_schema  = '{schema_use}'
    AND v.volume_name    = '{volume_name}'
  ORDER BY length(e.url) DESC   -- pick the most specific matching location
  LIMIT 1
""")

In [0]:
storage_location_str = volume_details.collect()[0]["storage_location"]
storage_location_str

In [0]:
# -------------------------------------------------------------------
# Helper UDF to build HL7 ADT messages for a sequence of events
# -------------------------------------------------------------------

@F.udf("string")
def build_hl7_adt_message(
    msg_datetime,
    event_type,
    patient_id,
    visit_id,
    last_name,
    first_name,
    dob,
    sex,
    ssn,
    street_address,
    city,
    state,
    zipcode,
    country,
    phone,
    nok_first_name,
    nok_last_name,
    nok_phone,
    nok_relationship,
    insurance_company,
    insurance_company_id,
    insurance_id,
    insurance_group,
    insurance_plan_id,
    insurance_plan_type,
    insurance_effective_date,
    insurance_expiration_date,
    patient_class,
    location,
    attending_id,
    attending_last,
    attending_first,
    discharge_disposition
):
    field_sep = "|"
    comp_sep = "^"
    rep_sep = "~"
    esc_char = "\\"
    subcomp_sep = "&"

    msg_ctrl_id = f"{patient_id}{visit_id}{msg_datetime}"

    # MSH segment
    msh = [
        "MSH",
        "^~\\&",
        sending_app,
        sending_facility,
        receiving_app,
        receiving_facility,
        msg_datetime,
        "",
        f"ADT^{event_type}",
        msg_ctrl_id,
        "P",
        hl7_version
    ]
    msh_seg = field_sep.join(msh)

    # EVN segment
    evn = [
        "EVN",
        event_type,
        msg_datetime
    ]
    evn_seg = field_sep.join(evn)

    # PID segment with address in PID-11
    pid = [
        "PID",
        "1",
        "",
        f"{patient_id}^^^DBXMRN^MR",
        "",
        f"{last_name}{comp_sep}{first_name}^^^^",
        "",
        dob,
        sex,
        "",
        # PID-11: Address (street^city^state^zip^country)
        f"{street_address}{comp_sep}{city}{comp_sep}{state}{comp_sep}{zipcode}{comp_sep}{country}",
        "",
        "",
        phone,  # PID-13: Phone number - home
        "",
        "",
        "",
        "",
        "",
        ssn,  # PID-19: SSN
    ]
    pid_seg = field_sep.join(pid)

    # NK1 segment - Next of Kin
    nk1 = [
        "NK1",
        "1",  # Set ID
        f"{nok_last_name}{comp_sep}{nok_first_name}^^^^",  # Name
        nok_relationship,  # Relationship (SPO=Spouse, CHD=Child, SIB=Sibling)
        "",
        nok_phone,  # Contact phone
    ]
    nk1_seg = field_sep.join(nk1)

    # PV1 segment
    pv1 = [
        "PV1",
        "1",
        patient_class,
        f"{location}^^^^^^^^^DBXFAC",
        "",
        "",
        "",
        "",
        "",
        "",
        f"{attending_id}{comp_sep}{attending_last}{comp_sep}{attending_first}^^^^",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        visit_id,
    ]
    # Add discharge disposition for A03 events
    if event_type == "A03" and discharge_disposition:
        if len(pv1) < 36:
            pv1 += [""] * (36 - len(pv1))
        pv1[35] = discharge_disposition

    pv1_seg = field_sep.join(pv1)

    # GT1 segment - Guarantor (Patient is guarantor)
    gt1 = [
        "GT1",
        "1",  # Set ID
        "",  # Guarantor number
        f"{last_name}{comp_sep}{first_name}^^^^",  # Guarantor name (same as patient)
        "",  # Guarantor spouse name
        f"{street_address}{comp_sep}{city}{comp_sep}{state}{comp_sep}{zipcode}{comp_sep}{country}",  # GT1-5: Guarantor address
        phone,  # GT1-6: Guarantor phone - home
        "",  # GT1-7: Guarantor phone - business
        dob,  # GT1-8: Guarantor DOB
        sex,  # GT1-9: Guarantor sex
        "",  # GT1-10: Guarantor type
        "SEL",  # GT1-11: Guarantor relationship (self)
        ssn,  # GT1-12: Guarantor SSN
    ]
    gt1_seg = field_sep.join(gt1)

    # IN1 segment - Primary Insurance (Enhanced)
    in1 = [
        "IN1",
        "1",  # Set ID
        insurance_plan_id,  # IN1-2: Insurance plan ID
        insurance_company_id,  # IN1-3: Insurance company ID
        insurance_company,  # IN1-4: Insurance company name
        f"{street_address}{comp_sep}{city}{comp_sep}{state}{comp_sep}{zipcode}{comp_sep}{country}",  # IN1-5: Insurance company address
        "",  # IN1-6: Insurance co contact person
        "",  # IN1-7: Insurance co phone number
        insurance_group,  # IN1-8: Group number
        "",  # IN1-9: Group name
        "",  # IN1-10: Insured's group emp ID
        "",  # IN1-11: Insured's group emp name
        insurance_effective_date,  # IN1-12: Plan effective date
        insurance_expiration_date,  # IN1-13: Plan expiration date
        "",  # IN1-14: Authorization information
        insurance_plan_type,  # IN1-15: Plan type (HMO, PPO, POS)
        f"{last_name}{comp_sep}{first_name}^^^^",  # IN1-16: Name of insured
        "SEL",  # IN1-17: Insured's relationship to patient (self)
        dob,  # IN1-18: Insured's DOB
        f"{street_address}{comp_sep}{city}{comp_sep}{state}{comp_sep}{zipcode}{comp_sep}{country}",  # IN1-19: Insured's address
        "",  # IN1-20: Assignment of benefits
        "",  # IN1-21: Coordination of benefits
        "",  # IN1-22: Coord of ben priority
        "",  # IN1-23: Notice of admission flag
        "",  # IN1-24: Notice of admission date
        "",  # IN1-25: Report of eligibility flag
        "",  # IN1-26: Report of eligibility date
        "",  # IN1-27: Release information code
        "",  # IN1-28: Pre-admit cert
        "",  # IN1-29: Verification date/time
        "",  # IN1-30: Verification by
        "",  # IN1-31: Type of agreement code
        "",  # IN1-32: Billing status
        "",  # IN1-33: Lifetime reserve days
        "",  # IN1-34: Delay before L.R. day
        "",  # IN1-35: Company plan code
        insurance_id,  # IN1-36: Policy number
    ]
    in1_seg = field_sep.join(in1)

    # IN2 segment - Secondary Insurance Information
    in2 = [
        "IN2",
        "",  # IN2-1: Insured's employee ID
        ssn,  # IN2-2: Insured's social security number
        "",  # IN2-3: Insured's employer name
        "",  # IN2-4: Employer information data
        "",  # IN2-5: Mail claim party
        "",  # IN2-6: Medicare health ins card number
        "",  # IN2-7: Medicaid case name
        "",  # IN2-8: Medicaid case number
        "",  # IN2-9: Military sponsor name
        "",  # IN2-10: Military ID number
        "",  # IN2-11: Dependent of military recipient
        "",  # IN2-12: Military organization
        "",  # IN2-13: Military station
        "",  # IN2-14: Military service
        "",  # IN2-15: Military rank/grade
        "",  # IN2-16: Military status
        "",  # IN2-17: Military retire date
        "",  # IN2-18: Military non-avail cert on file
        "",  # IN2-19: Baby coverage
        "",  # IN2-20: Combine baby bill
        "",  # IN2-21: Blood deductible
        "",  # IN2-22: Special coverage approval name
        "",  # IN2-23: Special coverage approval title
        "",  # IN2-24: Non-covered insurance code
        patient_id,  # IN2-25: Payor ID
        "SEL",  # IN2-26: Relationship to the patient (self)
    ]
    in2_seg = field_sep.join(in2)

    # Build complete message with all segments
    msg = "\r".join([msh_seg, evn_seg, pid_seg, nk1_seg, pv1_seg, gt1_seg, in1_seg, in2_seg])

    return msg

In [0]:
# -------------------------------------------------------------------
# 1) Generate base patient dimension (synthetic with Faker)
# -------------------------------------------------------------------
from faker import Faker
import random
import string
from datetime import datetime, timedelta

# UDFs to generate realistic patient data using Faker
@F.udf("string")
def generate_first_name(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.first_name()

@F.udf("string")
def generate_last_name(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.last_name()

@F.udf("string")
def generate_dob(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    # Generate DOB for adults aged 18-90
    dob = fake.date_of_birth(minimum_age=18, maximum_age=90)
    return dob.strftime("%Y%m%d")

@F.udf("string")
def generate_patient_id(seed_val, first_name, last_name):
    # Create patient ID: first 2 letters of last name + first letter of first name + 6 digits
    prefix = (last_name[:2] + first_name[:1]).upper()
    # Use seed to generate consistent number
    random.seed(seed_val)
    number = str(random.randint(100000, 999999))
    return f"{prefix}{number}"

@F.udf("string")
def generate_visit_id(seed_val):
    # Generate realistic visit ID: VIS + YYMMDD + 4-character alphanumeric
    fake = Faker()
    Faker.seed(seed_val)
    random.seed(seed_val)
    
    # Get a date component (using current year)
    visit_date = fake.date_between(start_date='-30d', end_date='today')
    date_str = visit_date.strftime("%y%m%d")
    
    # Generate 4-character alphanumeric suffix
    chars = string.ascii_uppercase + string.digits
    suffix = ''.join(random.choice(chars) for _ in range(4))
    
    return f"VIS{date_str}{suffix}"

@F.udf("string")
def generate_ssn(seed_val):
    # Generate fake SSN: ###-##-####
    random.seed(seed_val)
    area = str(random.randint(100, 899))  # Avoid certain ranges
    group = str(random.randint(10, 99))
    serial = str(random.randint(1000, 9999))
    return f"{area}-{group}-{serial}"

@F.udf("string")
def generate_street_address(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.street_address()

@F.udf("string")
def generate_city(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.city()

@F.udf("string")
def generate_state(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.state_abbr()

@F.udf("string")
def generate_zipcode(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    return fake.zipcode()

@F.udf("string")
def generate_phone(seed_val):
    fake = Faker()
    Faker.seed(seed_val)
    # Format: ###-###-####
    phone = fake.phone_number()
    # Clean up to basic format
    digits = ''.join(filter(str.isdigit, phone))[:10]
    if len(digits) == 10:
        return f"{digits[0:3]}-{digits[3:6]}-{digits[6:10]}"
    return phone

@F.udf("string")
def generate_insurance_effective_date(seed_val):
    # Events occur in January 2026, so generate effective dates 6-18 months before
    # This ensures recent, realistic coverage that's valid during events
    event_base = datetime(2026, 1, 1)
    random.seed(seed_val)
    days_before = random.randint(180, 540)  # 6-18 months before events
    effective_date = event_base - timedelta(days=days_before)
    return effective_date.strftime("%Y%m%d")

@F.udf("string")
def generate_insurance_expiration_date(seed_val, effective_date_str):
    # Generate expiration that ensures coverage through at least mid-2026
    # Set expiration 18-30 months after effective date
    try:
        effective_date = datetime.strptime(effective_date_str, "%Y%m%d")
        event_base = datetime(2026, 1, 1)
        random.seed(seed_val)
        
        # Calculate minimum months needed to cover events (from effective to mid-2026)
        months_to_events = ((event_base.year - effective_date.year) * 12 + 
                           (event_base.month - effective_date.month))
        # Add 6-12 months buffer beyond events
        min_coverage_months = months_to_events + 6
        max_coverage_months = months_to_events + 18
        
        # Generate expiration date
        coverage_days = random.randint(min_coverage_months * 30, max_coverage_months * 30)
        expiration_date = effective_date + timedelta(days=coverage_days)
        
        return expiration_date.strftime("%Y%m%d")
    except:
        return ""

patients_df = (
    spark.range(0, num_patients)
    .withColumnRenamed("id", "patient_index")
    .withColumn("seed", (F.col("patient_index") + 12345).cast("int"))  # Add offset for variety
    .withColumn("first_name", generate_first_name(F.col("seed")))
    .withColumn("last_name", generate_last_name(F.col("seed")))
    .withColumn("dob", generate_dob(F.col("seed")))
    .withColumn("patient_id", generate_patient_id(F.col("seed"), F.col("first_name"), F.col("last_name")))
    .withColumn("visit_id", generate_visit_id(F.col("seed")))
    .withColumn("sex", F.when((F.col("patient_index") % 2) == 0, "M").otherwise("F"))
    .withColumn("ssn", generate_ssn(F.col("seed") + 5))
    # Address fields
    .withColumn("street_address", generate_street_address(F.col("seed")))
    .withColumn("city", generate_city(F.col("seed") + 1))
    .withColumn("state", generate_state(F.col("seed") + 2))
    .withColumn("zipcode", generate_zipcode(F.col("seed") + 3))
    .withColumn("country", F.lit("USA"))
    .withColumn("phone", generate_phone(F.col("seed") + 4))
    # Next of kin fields (using offset seed for different person)
    .withColumn("nok_first_name", generate_first_name(F.col("seed") + 1000))
    .withColumn("nok_last_name", generate_last_name(F.col("seed") + 1001))
    .withColumn("nok_phone", generate_phone(F.col("seed") + 1002))
    .withColumn("nok_relationship", F.when((F.col("patient_index") % 3) == 0, "SPO").when((F.col("patient_index") % 3) == 1, "CHD").otherwise("SIB"))  # Spouse, Child, Sibling
    # Insurance fields
    .withColumn("insurance_company", F.when((F.col("patient_index") % 4) == 0, "BLUE CROSS").when((F.col("patient_index") % 4) == 1, "AETNA").when((F.col("patient_index") % 4) == 2, "CIGNA").otherwise("UNITED HEALTHCARE"))
    .withColumn("insurance_company_id", F.when((F.col("patient_index") % 4) == 0, "BCBS").when((F.col("patient_index") % 4) == 1, "AETN").when((F.col("patient_index") % 4) == 2, "CIGN").otherwise("UNHC"))
    .withColumn("insurance_id", F.concat(F.lit("INS"), F.lpad((F.col("patient_index") + 10000).cast("string"), 9, "0")))
    .withColumn("insurance_group", F.concat(F.lit("GRP"), F.lpad((F.col("patient_index") % 100).cast("string"), 5, "0")))
    .withColumn("insurance_plan_id", F.concat(F.lit("PLAN"), F.lpad((F.col("patient_index") % 50).cast("string"), 4, "0")))
    .withColumn("insurance_plan_type", F.when((F.col("patient_index") % 3) == 0, "HMO").when((F.col("patient_index") % 3) == 1, "PPO").otherwise("POS"))  # HMO, PPO, POS
    .withColumn("insurance_effective_date", generate_insurance_effective_date(F.col("seed") + 6))
    .withColumn("insurance_expiration_date", generate_insurance_expiration_date(F.col("seed") + 7, F.col("insurance_effective_date")))
    .select(
        "patient_index", "patient_id", "visit_id", "last_name", "first_name", "dob", "sex", "ssn",
        "street_address", "city", "state", "zipcode", "country", "phone",
        "nok_first_name", "nok_last_name", "nok_phone", "nok_relationship",
        "insurance_company", "insurance_company_id", "insurance_id", "insurance_group",
        "insurance_plan_id", "insurance_plan_type", "insurance_effective_date", "insurance_expiration_date"
    )
)

In [0]:
display(patients_df, 10)

In [0]:
# -------------------------------------------------------------------
# 2) Event plan per patient with realistic event types
# -------------------------------------------------------------------
event_schema = T.ArrayType(
    T.StructType([
        T.StructField("event_order", T.IntegerType(), False),
        T.StructField("event_type", T.StringType(), False)
    ])
)

@F.udf(event_schema)
def build_event_plan(random_int):
    events = []
    order = 0
    
    # Determine patient journey type based on random value
    journey_type = random_int % 10
    
    if journey_type == 0:
        # Journey 1: ER registration -> Admit -> Discharge
        events.append({"event_order": order, "event_type": "A04"})
        order += 1
        events.append({"event_order": order, "event_type": "A06"})  # Convert to inpatient
        order += 1
        # Add some updates
        for _ in range((random_int // 10) % 3):
            events.append({"event_order": order, "event_type": "A08"})
            order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    elif journey_type == 1:
        # Journey 2: Direct admit -> Transfer -> Discharge
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        # Add transfers
        for _ in range((random_int // 20) % 2 + 1):
            events.append({"event_order": order, "event_type": "A02"})
            order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    elif journey_type == 2:
        # Journey 3: Admit -> Cancel admit (patient left/error)
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A11"})  # Cancel admit
        
    elif journey_type == 3:
        # Journey 4: Admit -> Transfer planned -> Cancel transfer -> Eventually transfer -> Discharge
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A02"})
        order += 1
        events.append({"event_order": order, "event_type": "A12"})  # Cancel transfer
        order += 1
        events.append({"event_order": order, "event_type": "A02"})  # Actual transfer
        order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    elif journey_type == 4:
        # Journey 5: Admit -> Discharge planned -> Cancel discharge -> Eventually discharge
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A08"})
        order += 1
        events.append({"event_order": order, "event_type": "A03"})
        order += 1
        events.append({"event_order": order, "event_type": "A13"})  # Cancel discharge
        order += 1
        events.append({"event_order": order, "event_type": "A08"})
        order += 1
        events.append({"event_order": order, "event_type": "A03"})  # Actual discharge
        
    elif journey_type == 5:
        # Journey 6: ER registration only (no admission)
        events.append({"event_order": order, "event_type": "A04"})
        order += 1
        # Maybe some updates
        if (random_int // 50) % 2 == 0:
            events.append({"event_order": order, "event_type": "A08"})
            
    elif journey_type == 6:
        # Journey 7: Inpatient to outpatient status change
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        events.append({"event_order": order, "event_type": "A08"})
        order += 1
        events.append({"event_order": order, "event_type": "A07"})  # Change to outpatient
        order += 1
        events.append({"event_order": order, "event_type": "A03"})
        
    else:
        # Journey 8-9: Standard admit with updates and discharge
        events.append({"event_order": order, "event_type": "A01"})
        order += 1
        
        # Add varying number of updates
        n_updates = (random_int % 4)
        for _ in range(n_updates):
            events.append({"event_order": order, "event_type": "A08"})
            order += 1
            
        # Maybe add transfers
        n_transfers = (random_int // 4) % 3
        for _ in range(n_transfers):
            events.append({"event_order": order, "event_type": "A02"})
            order += 1
            
        # Most patients get discharged
        if (random_int % 5 != 0):
            events.append({"event_order": order, "event_type": "A03"})

    return events

events_planned = (
    patients_df
    .withColumn("rand", (F.rand() * 1000).cast("int"))
    .withColumn("events", build_event_plan(F.col("rand")))
    .select(
        "patient_id", "visit_id", "last_name", "first_name", "dob", "sex", "ssn",
        "street_address", "city", "state", "zipcode", "country", "phone",
        "nok_first_name", "nok_last_name", "nok_phone", "nok_relationship",
        "insurance_company", "insurance_company_id", "insurance_id", "insurance_group",
        "insurance_plan_id", "insurance_plan_type", "insurance_effective_date", "insurance_expiration_date",
        "events"
    )
)

events_exploded = (
    events_planned
    .withColumn("event", F.explode("events"))
    .select(
        "patient_id", "visit_id", "last_name", "first_name", "dob", "sex", "ssn",
        "street_address", "city", "state", "zipcode", "country", "phone",
        "nok_first_name", "nok_last_name", "nok_phone", "nok_relationship",
        "insurance_company", "insurance_company_id", "insurance_id", "insurance_group",
        "insurance_plan_id", "insurance_plan_type", "insurance_effective_date", "insurance_expiration_date",
        F.col("event.event_order").alias("event_order"),
        F.col("event.event_type").alias("event_type")
    )
)

In [0]:
# -------------------------------------------------------------------
# 3) Add timing and visit context
# -------------------------------------------------------------------
base_ts = F.to_timestamp(F.lit("2026-01-01 00:00:00"), "yyyy-MM-dd HH:mm:ss")

events_timed = (
    events_exploded
    .withColumn("hours_offset", (F.col("event_order") * 2 + (F.hash(F.col("patient_id")) % 24)).cast("int"))
    .withColumn(
        "event_datetime",
        F.date_format(
            base_ts + F.expr("MAKE_INTERVAL(0, 0, 0, 0, hours_offset, 0, 0)"),
            "yyyyMMddHHmmss"
        )
    )
    .withColumn(
        "patient_class",
        F.when(F.col("event_type") == "A04", "E")  # Emergency/ER registration
         .when(F.col("event_type") == "A06", "I")  # Converting to inpatient
         .when(F.col("event_type") == "A07", "O")  # Converting to outpatient
         .when(F.col("event_type").isin("A01", "A02", "A03"), "I")  # Inpatient events
         .when(F.col("event_type").isin("A11", "A12", "A13"), "I")  # Cancellations
         .otherwise("O")  # Updates default to outpatient
    )
    .withColumn(
        "location",
        F.concat(
            F.when(F.col("event_type") == "A04", F.lit("ER"))
             .otherwise(F.lit("WARD")),
            F.lpad((F.hash(F.col("patient_id")) % 10).cast("int").cast("string"), 2, "0"),
            F.lit("-ROOM"),
            F.lpad((F.hash(F.col("visit_id")) % 20).cast("int").cast("string"), 2, "0")
        )
    )
    .withColumn("attending_id", F.concat(F.lit("DR"), F.lpad((F.hash(F.col("patient_id")) % 50).cast("int").cast("string"), 3, "0")))
    .withColumn("attending_last", F.concat(F.lit("DOC"), F.lpad((F.hash(F.col("patient_id")) % 50).cast("int").cast("string"), 3, "0")))
    .withColumn("attending_first", F.lit("ATTENDING"))
    .withColumn(
        "discharge_disposition",
        F.when(F.col("event_type") == "A03",
               F.when((F.hash(F.col("patient_id")) % 2) == 0, "01").otherwise("03")
              ).otherwise("")
    )
    .drop("hours_offset")
)

In [0]:
display(events_timed)

In [0]:
# -------------------------------------------------------------------
# 4) Build HL7 message text
# -------------------------------------------------------------------
messages_df = (
    events_timed
    .withColumn(
        "hl7_message",
        build_hl7_adt_message(
            F.col("event_datetime"),
            F.col("event_type"),
            F.col("patient_id"),
            F.col("visit_id"),
            F.col("last_name"),
            F.col("first_name"),
            F.col("dob"),
            F.col("sex"),
            F.col("ssn"),
            F.col("street_address"),
            F.col("city"),
            F.col("state"),
            F.col("zipcode"),
            F.col("country"),
            F.col("phone"),
            F.col("nok_first_name"),
            F.col("nok_last_name"),
            F.col("nok_phone"),
            F.col("nok_relationship"),
            F.col("insurance_company"),
            F.col("insurance_company_id"),
            F.col("insurance_id"),
            F.col("insurance_group"),
            F.col("insurance_plan_id"),
            F.col("insurance_plan_type"),
            F.col("insurance_effective_date"),
            F.col("insurance_expiration_date"),
            F.col("patient_class"),
            F.col("location"),
            F.col("attending_id"),
            F.col("attending_last"),
            F.col("attending_first"),
            F.col("discharge_disposition")
        )
    )
    .select(
        "patient_id",
        "visit_id",
        "event_order",
        "event_type",
        "event_datetime",
        "hl7_message"
    )
    .orderBy("patient_id", "event_order")
)

In [0]:
display(messages_df, 10)

In [0]:
storage_location_str

In [0]:
# -------------------------------------------------------------------
# 5) Write as Parquet, then post-process to individual .hl7 files
# -------------------------------------------------------------------
# Two-stage approach for performance:
#   Stage 1: Write to Parquet in parallel (fast distributed write)
#   Stage 2: Read Parquet and split into individual .hl7 files (sequential)
#   Stage 3: Create 10 sample .txt files for testing
#
# Note: Threading on driver was tested but is slower due to dbutils.fs serialization.
#       Sequential writes are optimal for this use case.

import time

# Define subdirectories
parquet_stage_path = f"{output_path}/parquet_stage"
hl7_output_path = f"{output_path}/hl7_files"
txt_output_path = f"{output_path}/txt_files"

print(f"Stage 1: Writing to Parquet (parallel)...")
print(f"  Location: {parquet_stage_path}")

# Add filename column
messages_with_filename = messages_df.withColumn(
    "filename",
    F.concat(
        F.col("patient_id"),
        F.lit("_"),
        F.col("visit_id"),
        F.lit("_"),
        F.col("event_type"),
        F.lit("_"),
        F.col("event_datetime"),
        F.lit(".hl7")
    )
).select("filename", "hl7_message")

# Write as Parquet (distributed write - fast!)
messages_with_filename.write.mode("overwrite").parquet(parquet_stage_path)

print("✓ Stage 1 complete: Parquet written in parallel")

# Stage 2: Post-process Parquet to individual .hl7 files
print(f"\nStage 2: Post-processing to individual .hl7 files...")
print(f"  Output location: {hl7_output_path}")

# Read back from Parquet
messages_from_parquet = spark.read.parquet(parquet_stage_path)

# Collect and write individual files
messages_list = messages_from_parquet.collect()

print(f"  Writing {len(messages_list)} individual .hl7 files...")

# Ensure output directory exists
dbutils.fs.mkdirs(hl7_output_path)

# Write each message as individual file (sequential is fastest for UC volumes)
start_time = time.time()

for i, row in enumerate(messages_list, 1):
    filename = row["filename"]
    message = row["hl7_message"]
    file_path = f"{hl7_output_path}/{filename}"
    
    dbutils.fs.put(file_path, message, overwrite=True)
    
    # Show progress every 100 files
    if i % 100 == 0:
        print(f"    Progress: {i}/{len(messages_list)} files written...")

end_time = time.time()
write_time = end_time - start_time

print(f"\n✓ Stage 2 complete: {len(messages_list)} individual HL7 files created")
print(f"  Write time: {write_time:.1f}s ({len(messages_list)/write_time:.1f} files/sec)")

# Stage 3: Create 10 sample .txt files
print(f"\nStage 3: Creating 10 sample .txt files...")
print(f"  Output location: {txt_output_path}")

# Ensure txt output directory exists
dbutils.fs.mkdirs(txt_output_path)

# Take first 10 files and write them with .txt extension
for i, row in enumerate(messages_list[:10], 1):
    filename = row["filename"].replace(".hl7", ".txt")
    message = row["hl7_message"]
    file_path = f"{txt_output_path}/{filename}"
    
    dbutils.fs.put(file_path, message, overwrite=True)

print(f"✓ Stage 3 complete: 10 sample .txt files created")

# Clean up Parquet staging area
print(f"\nCleaning up staging area...")
dbutils.fs.rm(parquet_stage_path, recurse=True)
print(f"✓ Removed staging directory")

print(f"\n{'='*70}")
print(f"✓ SUCCESS: Wrote {len(messages_list)} HL7 files + 10 sample TXT files")
print(f"  Base path: {output_path}")
print(f"  HL7 files: {hl7_output_path} ({len(messages_list)} files)")
print(f"  TXT files: {txt_output_path} (10 files)")
print(f"  S3 Location: {storage_location_str}/{relative_path}/")
print(f"  Format: <patient_id>_<visit_id>_<event_type>_<timestamp>.[hl7|txt]")
print(f"  Performance: {write_time:.1f}s total, {len(messages_list)/write_time:.1f} files/sec")
print(f"{'='*70}")